# PubMedCLIP Baseline

 image-only baseline notebook. Uses train-only CUI vocabulary with minimum frequency 5 and maximum vocabulary cap 2048 for evaluation. Retrieval ranking uses visual similarity only; CUI labels are used only after ranking for metrics.

In [ ]:
import os, re, json, math, random, time, warnings, subprocess, sys, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from transformers import CLIPModel, CLIPProcessor
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "sentencepiece"])
    from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore")


DATA_ROOT = os.environ.get("DATA_ROOT", "../data/roco_v2")
OUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "pubmedclip_roco_retrieval"))

CFG = {
    "seed": 42,
    "random_seeds": [42, 123, 2025],


    "model_id": "flaviagiammarino/pubmed-clip-vit-base-patch32",


    "image_size": 224,


    "epochs": 30,
    "head_batch_size": 512,
    "cache_batch_size": 128,
    "embed_batch_size": 512,
    "num_workers": 2,


    "lr": 2e-4,
    "weight_decay": 1e-4,
    "dropout": 0.20,
    "proj_hidden_dim": 512,
    "proj_out_dim": 512,


    "max_train_cuis": 2048,
    "min_cui_freq": 5,


    "max_train_samples": None,
    "max_valid_samples": None,
    "max_test_samples": None,


    "eval_split": "test",
    "retrieval_pool": "test",
    "ks_main": [5, 10, 50],
    "ks_curve": list(range(1, 51)),
    "relevance_threshold": 0.0,
    "eval_chunk_size": 256,
    "topk_max": 50,


    "train": False,
    "amp": True,
    "use_cached_features": True,
    "use_trained_projection_for_retrieval": False,
}

os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Output directory:", OUT_DIR)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(CFG, f, indent=2)


def find_data_root(preferred):
    if Path(preferred).exists():
        return preferred
    print("DATA_ROOT not found. Searching DATA_SEARCH_DIR ...")
    candidates = []
    for root, dirs, files in os.walk(os.environ.get("DATA_SEARCH_DIR", "../data")):
        if {"train_concepts.csv", "valid_concepts.csv", "test_concepts.csv"}.issubset(set(files)):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError("Could not find folder with train/valid/test_concepts.csv")
    return candidates[0]

DATA_ROOT = find_data_root(DATA_ROOT)
print("Using DATA_ROOT:", DATA_ROOT)

CUI_RE = re.compile(r"C\d{7}", re.IGNORECASE)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def read_csv_flexible(path):
    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
            if df.shape[1] >= 1:
                return df
        except Exception:
            pass
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def build_image_map(split):
    split_dir = Path(DATA_ROOT) / split
    mp = {}
    for p in split_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            mp[p.stem] = str(p)
            mp[p.name] = str(p)
    return mp

def infer_id_col(df):
    for word in ["image", "file", "filename", "name", "id", "uid"]:
        for c in df.columns:
            if word in c.lower():
                return c
    return df.columns[0]

def extract_cuis_from_row(row):
    txt = " ".join([str(x) for x in row.values])
    return sorted(set([x.upper() for x in CUI_RE.findall(txt)]))

def load_split(split):
    csv_path = Path(DATA_ROOT) / f"{split}_concepts.csv"
    manual_path = Path(DATA_ROOT) / f"{split}_concepts_manual.csv"
    path = csv_path if csv_path.exists() else manual_path
    if not path.exists():
        raise FileNotFoundError(f"No concept CSV found for split={split}")

    df_raw = read_csv_flexible(path)
    id_col = infer_id_col(df_raw)
    img_map = build_image_map(split)

    rows, missing_path, no_cui = [], 0, 0
    for _, row in df_raw.iterrows():
        raw_id = str(row[id_col]).strip()
        base = Path(raw_id).stem

        img_path = None
        for key in [raw_id, base, raw_id + ".jpg", base + ".jpg", raw_id + ".png", base + ".png"]:
            if key in img_map:
                img_path = img_map[key]
                break

        if img_path is None:
            missing_path += 1
            continue

        cuis = extract_cuis_from_row(row)
        if len(cuis) == 0:
            no_cui += 1

        rows.append({"image_id": base, "path": img_path, "cuis": cuis, "n_cuis": len(cuis)})

    out = pd.DataFrame(rows)
    print(f"{split}: raw={len(df_raw)} usable={len(out)} missing_path={missing_path} no_cui={no_cui}")
    return out

def maybe_cap(df, max_n, seed):
    if max_n is not None and len(df) > int(max_n):
        return df.sample(n=int(max_n), random_state=seed).reset_index(drop=True)
    return df.reset_index(drop=True)

train_df = load_split("train")
valid_df = load_split("valid")
test_df = load_split("test")

train_df = train_df[train_df["n_cuis"] > 0].reset_index(drop=True)
valid_df = valid_df[valid_df["n_cuis"] > 0].reset_index(drop=True)
test_df = test_df[test_df["n_cuis"] > 0].reset_index(drop=True)

train_df = maybe_cap(train_df, CFG["max_train_samples"], CFG["seed"])
valid_df = maybe_cap(valid_df, CFG["max_valid_samples"], CFG["seed"])
test_df = maybe_cap(test_df, CFG["max_test_samples"], CFG["seed"])

overview = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "images": [len(train_df), len(valid_df), len(test_df)],
    "mean_cuis_per_image": [train_df.n_cuis.mean(), valid_df.n_cuis.mean(), test_df.n_cuis.mean()],
    "median_cuis_per_image": [train_df.n_cuis.median(), valid_df.n_cuis.median(), test_df.n_cuis.median()],
})
overview.to_csv(os.path.join(OUT_DIR, "dataset_overview.csv"), index=False)
display(overview)


plt.figure(figsize=(6,4))
plt.bar(overview["split"], overview["images"])
plt.title("Images per split")
plt.ylabel("Number of images")
savefig("01_images_per_split.png")

plt.figure(figsize=(6,4))
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    plt.hist(df["n_cuis"].values, bins=30, alpha=0.5, label=name)
plt.title("CUI count per image")
plt.xlabel("Number of CUIs")
plt.ylabel("Images")
plt.legend()
savefig("02_cui_count_histogram.png")

train_cui_counts = Counter(c for xs in train_df["cuis"] for c in xs)
top_cuis_df = pd.DataFrame(train_cui_counts.most_common(30), columns=["CUI", "count"])
top_cuis_df.to_csv(os.path.join(OUT_DIR, "top_30_train_cuis.csv"), index=False)

plt.figure(figsize=(10,5))
plt.bar(top_cuis_df["CUI"], top_cuis_df["count"])
plt.title("Top 30 CUI frequencies in training set")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Frequency")
savefig("03_top_cui_frequencies.png")


def make_label_space(train_df, max_labels, min_freq):
    counts = Counter(c for xs in train_df["cuis"] for c in xs)
    labels = [c for c, n in counts.most_common() if n >= min_freq][:max_labels]
    label_to_idx = {c: i for i, c in enumerate(labels)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label

label_to_idx, idx_to_label = make_label_space(train_df, CFG["max_train_cuis"], CFG["min_cui_freq"])
n_labels = len(label_to_idx)
print("Training CUI label count:", n_labels)

with open(os.path.join(OUT_DIR, "label_to_idx.json"), "w") as f:
    json.dump(label_to_idx, f, indent=2)

def to_label_indices(cuis):
    return [label_to_idx[c] for c in cuis if c in label_to_idx]

for df in [train_df, valid_df, test_df]:
    df["label_indices"] = df["cuis"].apply(to_label_indices)

train_cls_df = train_df[train_df["label_indices"].apply(len) > 0].reset_index(drop=True)
valid_cls_df = valid_df[valid_df["label_indices"].apply(len) > 0].reset_index(drop=True)
print("Train classifier samples:", len(train_cls_df))
print("Valid classifier samples:", len(valid_cls_df))


processor = CLIPProcessor.from_pretrained(CFG["model_id"])
image_processor = processor.image_processor


clip_mean = getattr(image_processor, "image_mean", [0.48145466, 0.4578275, 0.40821073])
clip_std = getattr(image_processor, "image_std", [0.26862954, 0.26130258, 0.27577711])

train_tfms = T.Compose([
    T.Resize((256, 256), interpolation=InterpolationMode.BICUBIC),
    T.RandomResizedCrop(CFG["image_size"], scale=(0.80, 1.0), interpolation=InterpolationMode.BICUBIC),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=clip_mean, std=clip_std),
])

eval_tfms = T.Compose([
    T.Resize((256, 256), interpolation=InterpolationMode.BICUBIC),
    T.CenterCrop(CFG["image_size"]),
    T.ToTensor(),
    T.Normalize(mean=clip_mean, std=clip_std),
])

class RocoImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["image_id"]


print("\nLoading PubMedCLIP:", CFG["model_id"])
pubmedclip = CLIPModel.from_pretrained(CFG["model_id"]).to(device)
pubmedclip.eval()


for p in pubmedclip.parameters():
    p.requires_grad = False

@torch.no_grad()
def extract_pubmedclip_features(df, name, transform):
    feat_path = os.path.join(OUT_DIR, f"{name}_pubmedclip_raw_features.npy")
    ids_path = os.path.join(OUT_DIR, f"{name}_ids.csv")

    if CFG["use_cached_features"] and os.path.exists(feat_path) and os.path.exists(ids_path):
        arr = np.load(feat_path)
        ids = pd.read_csv(ids_path)["image_id"].astype(str).tolist()
        if len(arr) == len(df) and len(ids) == len(df):
            print(f"Loaded cached {name} features:", arr.shape)
            return arr.astype("float32"), ids
        else:
            print(f"Cache size mismatch for {name}. Recomputing...")

    loader = DataLoader(
        RocoImageDataset(df, transform),
        batch_size=CFG["cache_batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=True
    )

    feats, ids = [], []
    pubmedclip.eval()

    for imgs, image_ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
            out = pubmedclip.get_image_features(pixel_values=imgs)


            if isinstance(out, torch.Tensor):
                image_features = out
            elif hasattr(out, "image_embeds") and out.image_embeds is not None:
                image_features = out.image_embeds
            elif hasattr(out, "pooler_output") and out.pooler_output is not None:
                image_features = out.pooler_output
            elif hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
                image_features = out.last_hidden_state[:, 0]
            elif isinstance(out, (tuple, list)):
                image_features = None
                for item in out:
                    if isinstance(item, torch.Tensor):
                        image_features = item
                        break
                    if hasattr(item, "pooler_output") and item.pooler_output is not None:
                        image_features = item.pooler_output
                        break
                if image_features is None:
                    raise TypeError(f"Could not extract image features from output type: {type(out)}")
            else:
                raise TypeError(f"Could not extract image features from output type: {type(out)}")

            if image_features.ndim == 3:
                image_features = image_features[:, 0]


            if hasattr(pubmedclip, "visual_projection"):
                proj = pubmedclip.visual_projection
                if hasattr(proj, "in_features") and image_features.shape[-1] == proj.in_features:
                    image_features = proj(image_features)

            image_features = F.normalize(image_features.float(), dim=1)

        feats.append(image_features.cpu().numpy().astype("float32"))
        ids.extend(list(image_ids))

    feats = np.concatenate(feats, axis=0).astype("float32")
    np.save(feat_path, feats)
    pd.DataFrame({"image_id": ids}).to_csv(ids_path, index=False)
    print(f"Saved {name} features:", feats.shape)
    return feats, ids

train_raw, train_ids = extract_pubmedclip_features(train_cls_df, "train_cls", train_tfms)
valid_raw, valid_ids = extract_pubmedclip_features(valid_cls_df, "valid_cls", eval_tfms)


pubmedclip.to("cpu")
del pubmedclip
if device.type == "cuda":
    torch.cuda.empty_cache()


class FeatureCUIDataset(Dataset):
    def __init__(self, features, df, n_labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.df = df.reset_index(drop=True)
        self.n_labels = n_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = torch.zeros(self.n_labels, dtype=torch.float32)
        for j in self.df.iloc[idx]["label_indices"]:
            y[j] = 1.0
        return x, y

class PubMedCLIPRetrievalHead(nn.Module):
    def __init__(self, in_dim, n_labels, hidden_dim=512, out_dim=512, dropout=0.20):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim),
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(out_dim, n_labels)

    def forward(self, x, return_embedding=False):
        z = self.proj(x)
        z = F.normalize(z, dim=1)
        if return_embedding:
            return z
        return self.classifier(self.dropout(z))

in_dim = train_raw.shape[1]
model = PubMedCLIPRetrievalHead(
    in_dim=in_dim,
    n_labels=n_labels,
    hidden_dim=CFG["proj_hidden_dim"],
    out_dim=CFG["proj_out_dim"],
    dropout=CFG["dropout"],
).to(device)

train_loader = DataLoader(
    FeatureCUIDataset(train_raw, train_cls_df, n_labels),
    batch_size=CFG["head_batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    FeatureCUIDataset(valid_raw, valid_cls_df, n_labels),
    batch_size=CFG["head_batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


pos_counts = np.zeros(n_labels, dtype=np.float32)
for inds in train_cls_df["label_indices"]:
    pos_counts[inds] += 1
pos_weight = (len(train_cls_df) - pos_counts) / np.maximum(pos_counts, 1)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG["epochs"]))
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["amp"] and device.type == "cuda"))
CKPT_PATH = os.path.join(OUT_DIR, "best_pubmedclip_retrieval_head.pth")


def run_one_epoch(loader, train=True):
    model.train(train)
    total, n = 0.0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total += float(loss.item()) * x.size(0)
        n += x.size(0)

    return total / max(n, 1)

history = []
best_val = float("inf")

if CFG["train"]:
    print("\nTraining PubMedCLIP retrieval/classifier head...")
    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        tr_loss = run_one_epoch(train_loader, train=True)
        va_loss = run_one_epoch(valid_loader, train=False)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": tr_loss,
            "valid_loss": va_loss,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - t0,
        }
        history.append(row)
        print(f"Epoch {epoch:02d}/{CFG['epochs']} | train={tr_loss:.5f} | valid={va_loss:.5f} | time={row['seconds']:.1f}s")

        if va_loss < best_val:
            best_val = va_loss
            torch.save({
                "model": model.state_dict(),
                "cfg": CFG,
                "label_to_idx": label_to_idx,
                "best_val": best_val,
                "in_dim": in_dim,
            }, CKPT_PATH)
            print("  saved best checkpoint")

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)

    plt.figure(figsize=(6,4))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train")
    plt.plot(hist_df["epoch"], hist_df["valid_loss"], marker="o", label="valid")
    plt.title("PubMedCLIP head training and validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.legend()
    savefig("04_training_loss_curve.png")

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    print("Loaded best checkpoint:", CKPT_PATH)


split_to_df = {"train": train_df, "valid": valid_df, "test": test_df}
query_df = split_to_df[CFG["eval_split"]].reset_index(drop=True)
pool_df = split_to_df[CFG["retrieval_pool"]].reset_index(drop=True)

same_pool = (
    CFG["eval_split"] == CFG["retrieval_pool"] and
    len(query_df) == len(pool_df) and
    list(query_df["image_id"]) == list(pool_df["image_id"])
)
print("same query/pool:", same_pool)


print("\nReloading PubMedCLIP for query/pool feature extraction...")
pubmedclip = CLIPModel.from_pretrained(CFG["model_id"]).to(device)
pubmedclip.eval()
for p in pubmedclip.parameters():
    p.requires_grad = False

query_raw, query_ids = extract_pubmedclip_features(query_df, CFG["eval_split"] + "_query", eval_tfms)

if same_pool:
    pool_raw, pool_ids = query_raw, query_ids

    np.save(os.path.join(OUT_DIR, f"{CFG['retrieval_pool']}_pool_pubmedclip_raw_features.npy"), pool_raw)
    pd.DataFrame({"image_id": pool_ids}).to_csv(os.path.join(OUT_DIR, f"{CFG['retrieval_pool']}_pool_ids.csv"), index=False)
else:
    pool_raw, pool_ids = extract_pubmedclip_features(pool_df, CFG["retrieval_pool"] + "_pool", eval_tfms)

pubmedclip.to("cpu")
del pubmedclip
if device.type == "cuda":
    torch.cuda.empty_cache()

@torch.no_grad()
def project_embeddings(raw_features, name):
    emb_path = os.path.join(OUT_DIR, f"{name}_embeddings.npy")
    if CFG["use_cached_features"] and os.path.exists(emb_path):
        arr = np.load(emb_path)
        if len(arr) == len(raw_features):
            print(f"Loaded cached projected embeddings for {name}:", arr.shape)
            return arr.astype("float32")
    if not CFG["use_trained_projection_for_retrieval"]:
        embs = raw_features / np.maximum(np.linalg.norm(raw_features, axis=1, keepdims=True), 1e-8)
        np.save(emb_path, embs.astype("float32"))
        print(name, embs.shape)
        return embs.astype("float32")
    model.eval()
    all_embs = []
    x_tensor = torch.tensor(raw_features, dtype=torch.float32)
    loader = DataLoader(x_tensor, batch_size=CFG["embed_batch_size"], shuffle=False, num_workers=0)
    for x in loader:
        x = x.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(CFG["amp"] and device.type == "cuda")):
            z = model(x, return_embedding=True)
        all_embs.append(z.float().cpu().numpy().astype("float32"))
    embs = np.concatenate(all_embs, axis=0).astype("float32")
    np.save(emb_path, embs)
    print(name, embs.shape)
    return embs

query_embs = project_embeddings(query_raw, CFG["eval_split"] + "_query")
pool_embs = query_embs if same_pool else project_embeddings(pool_raw, CFG["retrieval_pool"] + "_pool")


def compute_topk_by_embedding(query_embs, pool_embs, topk, chunk_size, same_pool):
    q = torch.tensor(query_embs, dtype=torch.float32, device=device)
    p = torch.tensor(pool_embs, dtype=torch.float32, device=device)
    q = F.normalize(q, dim=1)
    p = F.normalize(p, dim=1)

    all_idx, all_sim = [], []

    for s in range(0, q.shape[0], chunk_size):
        e = min(s + chunk_size, q.shape[0])
        sim = q[s:e] @ p.T

        if same_pool:
            diag = torch.arange(s, e, device=device)
            sim[torch.arange(e - s, device=device), diag] = -1e9

        k_use = min(topk, p.shape[0] - 1 if same_pool else p.shape[0])
        vals, inds = torch.topk(sim, k=k_use, dim=1)
        all_idx.append(inds.cpu().numpy())
        all_sim.append(vals.cpu().numpy())

    return np.vstack(all_idx), np.vstack(all_sim)

topk_idx, topk_sim = compute_topk_by_embedding(
    query_embs, pool_embs,
    topk=CFG["topk_max"],
    chunk_size=CFG["eval_chunk_size"],
    same_pool=same_pool
)

np.save(os.path.join(OUT_DIR, "retrieval_topk_indices.npy"), topk_idx)
np.save(os.path.join(OUT_DIR, "retrieval_topk_similarities.npy"), topk_sim)

rows = []
for qi in range(len(query_df)):
    for r in range(topk_idx.shape[1]):
        pi = int(topk_idx[qi, r])
        rows.append({
            "query_image_id": query_df.iloc[qi]["image_id"],
            "rank": r + 1,
            "retrieved_image_id": pool_df.iloc[pi]["image_id"],
            "cosine_similarity": float(topk_sim[qi, r]),
        })
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "retrieval_top50.csv"), index=False)


def build_cui_vocab(*dfs):
    vocab = {}
    for df in dfs:
        for cuis in df["cuis"]:
            for c in cuis:
                if c not in vocab:
                    vocab[c] = len(vocab)
    return vocab

def build_csr(df, vocab):
    rows, cols = [], []
    for i, cuis in enumerate(df["cuis"]):
        for c in set(cuis):
            if c in vocab:
                rows.append(i)
                cols.append(vocab[c])
    data = np.ones(len(rows), dtype=np.float32)
    mat = csr_matrix((data, (rows, cols)), shape=(len(df), len(vocab)), dtype=np.float32)
    lengths = np.asarray(mat.sum(axis=1)).reshape(-1).astype(np.float32)
    return mat, lengths

cui_vocab = build_cui_vocab(query_df, pool_df)
query_csr, query_len = build_csr(query_df, cui_vocab)
pool_csr, pool_len = build_csr(pool_df, cui_vocab)
print("CUI vocab:", len(cui_vocab))


def dcg_at_k(gains):
    gains = np.asarray(gains, dtype=np.float32)
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))

def ap_at_k(rels, total_relevant, k):
    if total_relevant <= 0:
        return np.nan
    rels = np.asarray(rels[:k], dtype=np.float32)
    hits, precisions = 0, []
    for i, r in enumerate(rels, start=1):
        if r > 0:
            hits += 1
            precisions.append(hits / i)
    return float(np.sum(precisions) / min(total_relevant, k)) if precisions else 0.0

def rr_at_k(rels, k):
    rels = np.asarray(rels[:k], dtype=np.float32)
    hits = np.where(rels > 0)[0]
    return float(1.0 / (hits[0] + 1)) if len(hits) else 0.0

def summarize(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[~np.isnan(values)]
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    ci95 = float(1.96 * std / math.sqrt(n)) if n > 1 else 0.0
    return mean, std, ci95, n

def evaluate_metrics(query_csr, pool_csr, query_len, pool_len, pred_idx, ks, threshold, same_pool, chunk_size):
    maxk = max(ks)
    per_query = []

    for s in range(0, query_csr.shape[0], chunk_size):
        e = min(s + chunk_size, query_csr.shape[0])

        inter = query_csr[s:e].dot(pool_csr.T).toarray().astype(np.float32)
        union = query_len[s:e, None] + pool_len[None, :] - inter
        all_gains = np.divide(inter, np.maximum(union, 1e-8), out=np.zeros_like(inter), where=union > 0)

        if same_pool:
            for local_i, global_i in enumerate(range(s, e)):
                all_gains[local_i, global_i] = -1.0

        for local_i, global_i in enumerate(range(s, e)):
            top_idx = pred_idx[global_i, :maxk]
            pred_gains = all_gains[local_i, top_idx].copy()
            pred_gains[pred_gains < 0] = 0.0

            gains = all_gains[local_i].copy()
            gains[gains < 0] = 0.0
            total_relevant = int((gains > threshold).sum())
            ideal_sorted = np.sort(gains)[::-1]

            row = {
                "query_index": global_i,
                "query_image_id": query_df.iloc[global_i]["image_id"],
                "total_relevant_by_cui_jaccard": total_relevant,
            }

            for k in ks:
                gk = pred_gains[:k]
                rel = gk > threshold

                idcg = dcg_at_k(ideal_sorted[:k])
                row[f"CUI@{k}"] = dcg_at_k(gk) / idcg if idcg > 0 else 0.0
                row[f"Precision@{k}"] = float(rel.sum() / k)
                row[f"Recall@{k}"] = float(rel.sum() / total_relevant) if total_relevant > 0 else np.nan
                row[f"mAP@{k}"] = ap_at_k(rel, total_relevant, k)
                row[f"MRR@{k}"] = rr_at_k(rel, k)
                row[f"MeanJaccard@{k}"] = float(np.mean(gk))

            per_query.append(row)

    per_query = pd.DataFrame(per_query)

    summary = []
    for col in [c for c in per_query.columns if "@" in c]:
        mean, std, ci95, n = summarize(per_query[col].values)
        summary.append({
            "metric": col,
            "mean": mean,
            "std": std,
            "ci95": ci95,
            "n_queries_used": n,
            "mean±std": f"{mean:.4f} ± {std:.4f}",
            "mean±95%CI": f"{mean:.4f} ± {ci95:.4f}",
        })

    return per_query, pd.DataFrame(summary)

all_ks = sorted(set(CFG["ks_main"] + CFG["ks_curve"]))
per_query_metrics, metrics_summary = evaluate_metrics(
    query_csr, pool_csr, query_len, pool_len,
    topk_idx, all_ks,
    threshold=CFG["relevance_threshold"],
    same_pool=same_pool,
    chunk_size=CFG["eval_chunk_size"],
)

per_query_metrics.to_csv(os.path.join(OUT_DIR, "per_query_metrics.csv"), index=False)
metrics_summary.to_csv(os.path.join(OUT_DIR, "metrics_summary_all_k.csv"), index=False)

main_cols = []
for metric in ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]:
    for k in CFG["ks_main"]:
        main_cols.append(f"{metric}@{k}")

main_summary = metrics_summary[metrics_summary["metric"].isin(main_cols)].copy()
main_summary["order"] = main_summary["metric"].apply(lambda x: main_cols.index(x))
main_summary = main_summary.sort_values("order").drop(columns=["order"])
main_summary.to_csv(os.path.join(OUT_DIR, "metrics_summary_main.csv"), index=False)

print("\n========== MAIN RESULTS ==========")
display(main_summary[["metric", "mean±std", "mean±95%CI", "n_queries_used"]])

no_rel = int((per_query_metrics["total_relevant_by_cui_jaccard"] == 0).sum())
print(f"Queries with no relevant non-self image: {no_rel}/{len(per_query_metrics)}")


def metric_curve(summary, prefix):
    xs, means, cis = [], [], []
    for k in CFG["ks_curve"]:
        name = f"{prefix}@{k}"
        hit = summary[summary["metric"] == name]
        if len(hit):
            xs.append(k)
            means.append(float(hit["mean"].iloc[0]))
            cis.append(float(hit["ci95"].iloc[0]))
    return np.array(xs), np.array(means), np.array(cis)

x, y, ci = metric_curve(metrics_summary, "CUI")
plt.figure(figsize=(7,4))
plt.plot(x, y, marker="o", markersize=3)
plt.fill_between(x, y-ci, y+ci, alpha=0.2)
plt.title("PubMedCLIP CUI@K curve with 95% CI")
plt.xlabel("K")
plt.ylabel("CUI@K")
savefig("05_cui_at_k_curve.png")

xp, p, pci = metric_curve(metrics_summary, "Precision")
xr, r, rci = metric_curve(metrics_summary, "Recall")
plt.figure(figsize=(7,4))
plt.plot(xp, p, marker="o", markersize=3, label="Precision@K")
plt.fill_between(xp, p-pci, p+pci, alpha=0.2)
plt.plot(xr, r, marker="o", markersize=3, label="Recall@K")
plt.fill_between(xr, r-rci, r+rci, alpha=0.2)
plt.title("PubMedCLIP Precision@K and Recall@K")
plt.xlabel("K")
plt.ylabel("Score")
plt.legend()
savefig("06_precision_recall_at_k_curves.png")

plt.figure(figsize=(6,5))
plt.plot(r, p, marker="o", markersize=3)
for kk in [1, 5, 10, 20, 50]:
    if kk in xp:
        idx = list(xp).index(kk)
        plt.annotate(f"K={kk}", (r[idx], p[idx]))
plt.title("PubMedCLIP precision-recall tradeoff across K")
plt.xlabel("Recall@K")
plt.ylabel("Precision@K")
savefig("07_precision_recall_tradeoff.png")

bar_metrics = [f"CUI@{k}" for k in CFG["ks_main"]] + [f"Precision@{k}" for k in CFG["ks_main"]] + [f"Recall@{k}" for k in CFG["ks_main"]]
bar_df = main_summary[main_summary["metric"].isin(bar_metrics)].copy()
plt.figure(figsize=(10,5))
plt.bar(bar_df["metric"], bar_df["mean"], yerr=bar_df["std"], capsize=4)
plt.xticks(rotation=45, ha="right")
plt.title("PubMedCLIP main metrics: mean ± std across queries")
plt.ylabel("Score")
savefig("08_main_metrics_bar_mean_std.png")

bar_metrics2 = [f"mAP@{k}" for k in CFG["ks_main"]] + [f"MRR@{k}" for k in CFG["ks_main"]]
bar_df2 = main_summary[main_summary["metric"].isin(bar_metrics2)].copy()
plt.figure(figsize=(8,4))
plt.bar(bar_df2["metric"], bar_df2["mean"], yerr=bar_df2["std"], capsize=4)
plt.xticks(rotation=45, ha="right")
plt.title("PubMedCLIP mAP and MRR: mean ± std")
plt.ylabel("Score")
savefig("09_map_mrr_bar_mean_std.png")

plt.figure(figsize=(6,4))
plt.hist(topk_sim[:, 0], bins=40, alpha=0.8)
plt.title("PubMedCLIP top-1 cosine similarity distribution")
plt.xlabel("Top-1 cosine similarity")
plt.ylabel("Queries")
savefig("10_top1_similarity_distribution.png")


sample_n = min(3000, len(pool_embs))
rng = np.random.default_rng(CFG["seed"])
sample_idx = rng.choice(len(pool_embs), size=sample_n, replace=False)
sample_embs = pool_embs[sample_idx]

xy = PCA(n_components=2, random_state=CFG["seed"]).fit_transform(sample_embs)
clusters = KMeans(n_clusters=min(10, sample_n), random_state=CFG["seed"], n_init=10).fit_predict(sample_embs)

plt.figure(figsize=(7,6))
plt.scatter(xy[:, 0], xy[:, 1], c=clusters, s=8, alpha=0.8)
plt.title("PubMedCLIP projected embedding PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
savefig("11_embedding_pca_kmeans_clusters.png")


def load_img_for_plot(path, size=180):
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size))
    return img

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(1, len(a | b))

def plot_retrieval_grid(query_indices, k, name):
    n_rows = len(query_indices)
    n_cols = k + 1
    plt.figure(figsize=(2.1*n_cols, 2.3*n_rows))

    for rr, qi in enumerate(query_indices):
        qrow = query_df.iloc[qi]
        ax = plt.subplot(n_rows, n_cols, rr*n_cols + 1)
        ax.imshow(load_img_for_plot(qrow["path"]))
        ax.set_title("Query\n" + qrow["image_id"][-10:], fontsize=8)
        ax.axis("off")

        for j in range(k):
            pi = int(topk_idx[qi, j])
            prow = pool_df.iloc[pi]
            ax = plt.subplot(n_rows, n_cols, rr*n_cols + 2 + j)
            ax.imshow(load_img_for_plot(prow["path"]))
            jac = jaccard(qrow["cuis"], prow["cuis"])
            ax.set_title(f"Rank {j+1}\nJ={jac:.2f}", fontsize=8)
            ax.axis("off")

    savefig(name)

rand_q = rng.choice(len(query_df), size=min(6, len(query_df)), replace=False).tolist()
plot_retrieval_grid(rand_q, 5, "12_random_top5_retrieval_examples.png")

if "CUI@5" in per_query_metrics.columns:
    best_q = per_query_metrics.sort_values("CUI@5", ascending=False)["query_index"].head(6).tolist()
    worst_q = per_query_metrics.sort_values("CUI@5", ascending=True)["query_index"].head(6).tolist()
    plot_retrieval_grid(best_q, 5, "13_best_cui5_top5_retrieval_examples.png")
    plot_retrieval_grid(worst_q, 5, "14_worst_cui5_top5_retrieval_examples.png")
